In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import json
import socket
import re

from utils_mitgcm import *
from utils_eof import *

In [ ]:
def open_synthetic_ds_from_date(config, synthetic_datapath, date_str):
    datapath = f"{synthetic_datapath}_{date_str}"
    gridpath = config['gridpath']
    ref_date = config['ref_date']
    dt_mitgcm_results = config['dt']
    endian = config['endian']

    return config, open_mitgcm_ds(datapath, gridpath, ref_date, dt_mitgcm_results, endian)

In [ ]:
synthetic_datapath =  "/storage/alplakes_test/neuchatel_100m_EOF_synthetic/outputs"

In [ ]:
lake = 'neuchatel'
model = f'{lake}_2025'

In [ ]:
with open('../../../config.json', 'r') as file:
    config = json.load(file)[socket.gethostname()][model]
base_folder_path = os.path.dirname(config['datapath'])

# Extract rotary complex EOF from synthetic events

In [ ]:
time_window = range(2*24, 5*24)

In [ ]:
def extract_rotary_eof_synthetic_events(config, synthetic_datapath, date_str, time_window, base_folder_path):
    """

    Parameters
    ----------
    config: mitgcm config
    synthetic_datapath: folder in which the synthetic datasets are stored
    date_str: format "YYYY-MM-DD"
    time_window: time window on which the EOF is extracted
    base_folder_path: folder where the continuous simulation results are stored

    Returns
    ----------
    patterns_ds: xarray dataset containing the EOF patterns (u,v)
    """

    synth_config, ds_synth = open_synthetic_ds_from_date(config, synthetic_datapath, date_str)
    ds_synth, u_synth, v_synth, w_synth = align_coordinates(ds_synth)

    output_folder = os.path.join(base_folder_path, "eof", "synthetic_events", date_str)
    os.makedirs(output_folder, exist_ok=True)
    plt.close('all')

    # Figure 1: Synthetic velocities
    plt.figure(figsize=(15,5))
    u_synth.isel(XC=200, YC=40, Z=0).plot()
    plt.title(f"XC_idx=200, YC_idx=40, Z_idx=0")
    plt.ylabel("U [m/s]")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f"{date_str}_synthetic_velocities_timeserie.png"))

    # Figure 2: Synthetic velocities map
    ii = 48
    zz = 0
    subsetting_factor = 4
    plt.figure(figsize=(15,5))
    ds_synth.THETA.isel(time=ii, Z=zz).where(ds_synth.THETA.isel(time=ii, Z=zz)>0).plot()
    plt.quiver(u_synth.XC[::subsetting_factor], u_synth.YC[::subsetting_factor],
               u_synth.isel(time=ii, Z=zz)[::subsetting_factor,::subsetting_factor],v_synth.isel(time=ii, Z=zz)[::subsetting_factor,::subsetting_factor],
               scale=5)
    plt.title(f"Time {ii}h, depth layer {zz}")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f"{date_str}_synthetic_velocities_map.png"))

    # Extract EOF
    rotary = rotary_eof_xarray(u_synth.isel(time=time_window), v_synth.isel(time=time_window), 100*100, n_modes=5)
    mode=0

    # Figure 3 & 4: Pattern and amplitude of EOF
    print_and_save_figure_rotary_eof(mode, rotary, output_folder, date_str)

    # Save EOF
    EOF = rotary.EOF.isel(mode=mode)
    U_pattern = EOF.real
    V_pattern = EOF.imag
    patterns_ds = xr.Dataset(
        data_vars={
            "u_eof": U_pattern,
            "v_eof": V_pattern,
        },
        coords={
            "Z": U_pattern["Z"],
            "YC": U_pattern["YC"],
            "XC": U_pattern["XC"],
        },
        attrs={"description": "EOF velocity patterns (U and V components)"},
    )

    patterns_ds.to_netcdf(os.path.join(output_folder, f"{date_str}_eof.nc"))

    return patterns_ds

In [ ]:
# Extract and list the dates of the synthetic datasets
_all_subfolders = [
    name for name in os.listdir(os.path.dirname(synthetic_datapath))
    if os.path.isdir(os.path.join(os.path.dirname(synthetic_datapath), name))
]

_pat = re.compile(r"^outputs_(\d{4}-\d{2}-\d{2})$")

outputs_folders = [name for name in _all_subfolders if _pat.match(name)]
outputs_dates = [m.group(1) for name in outputs_folders for m in [_pat.match(name)]]

In [ ]:
for date_str in outputs_dates:
    extract_rotary_eof_synthetic_events(config, synthetic_datapath, date_str, time_window, base_folder_path)

# Further analyses

In [ ]:
date_str='2025-05-07'

synth_config, ds_synth = open_synthetic_ds_from_date(config, synthetic_datapath, date_str)
ds_synth, u_synth, v_synth, w_synth = align_coordinates(ds_synth)

rotary = rotary_eof_xarray(u_synth.isel(time=time_window), v_synth.isel(time=time_window), 100*100, n_modes=5)

In [ ]:
u_synth.isel(XC=200, YC=40, Z=0).plot()

In [ ]:
plt.figure(figsize=(8, 4))
rotary.variance_fraction.plot(marker='o', markersize=5)
plt.ylabel('Fraction of variance explained [-]')
plt.xlabel('Mode')
plt.title(f'Fraction of variance explained')
plt.tight_layout()

In [ ]:
zz = 0
subsetting_factor = 4
output_folder = os.path.join(base_folder_path, "eof", "synthetic_events", date_str, 'synth_velocities')
os.makedirs(output_folder, exist_ok=True)

for ii in range(86):
    plt.close('all')
    plt.figure(figsize=(15,5))
    ds_synth.THETA.isel(time=ii, Z=zz).where(ds_synth.THETA.isel(time=ii, Z=zz)>0).plot()
    plt.quiver(u_synth.XC[::subsetting_factor], u_synth.YC[::subsetting_factor],
               u_synth.isel(time=ii, Z=zz)[::subsetting_factor,::subsetting_factor],v_synth.isel(time=ii, Z=zz)[::subsetting_factor,::subsetting_factor],
               scale=5)
    plt.title(f"Time {ii}h, depth layer {zz}")
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f"synthetic_velocities_map_{ii}.png"))

In [ ]:
mode=0
figsize=(15, 6)
subsetting_factor=5
zz=0

EOF = rotary.EOF.isel(mode=mode)

# U and V components
U_pattern = EOF.real
V_pattern = EOF.imag

# Compute horizontal amplitude
amp = np.sqrt(U_pattern ** 2 + V_pattern ** 2)

# Plot quiver for a single depth slice
plt.figure(figsize=figsize)
amp.isel(Z=zz).plot(add_colorbar=False)
plt.quiver(rotary.XC[::subsetting_factor], rotary.YC[::subsetting_factor],
           U_pattern[zz, :, :][::subsetting_factor, ::subsetting_factor],
           V_pattern[zz, :, :][::subsetting_factor, ::subsetting_factor],
           scale=1e-5)
plt.gca().invert_yaxis()
plt.title(f'Rotary EOF mode {mode + 1}, depth level {zz}')
plt.xlabel('X')
plt.ylabel('Y')
plt.colorbar(label='Amplitude')
plt.tight_layout()

PC = rotary.PC.isel(mode=mode)
plt.figure(figsize=(10, 4))
plt.plot(rotary.time, PC, label='Amplitude')
plt.ylabel('Amplitude (sqrt(KE))')
plt.xlabel('Time')
plt.title(f'PC amplitude, mode {mode + 1}')
plt.grid()
plt.tight_layout()